# EP2 - Grid de hiperparâmetros do BERTimbau (Colab)Varre `learning_rate` x `épocas` no BERTimbau **base** (110M), que nunca foiajustado - os valores do EP1 (lr 2e-5, 4 épocas) foram escolhidos à mão.Protocolo **idêntico** ao local, para os números serem comparáveis:holdout 20% estratificado seed 42 **intocado**, validação 10% do dev para earlystopping pelo F1-macro, avaliação única no holdout.Cada configuração salva `probs_grid_<tag>.npz`, que eu testo no ensemble com oprotocolo de seleção honesta (split-half).**Antes de rodar:** `Ambiente de execução > Alterar o tipo > GPU` (L4 de preferência).Tempo estimado: ~25-40 min por configuração na L4.Referências locais no MESMO holdout:- BERT base 1 semente (config do EP1): **0,4500**- BERT base 3 sementes (ensemble): 0,4573- BERT + ComplementNB: **0,4760** ← alvo a superar- BERTimbau large: 0,4486 (não ajudou)

In [ ]:
!nvidia-smi
!pip install -q transformers accelerate openpyxl

In [ ]:
from google.colab import files
up = files.upload()   # selecione train.xlsx

In [ ]:
import numpy as np, pandas as pd, torch, json, os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, EarlyStoppingCallback,
                          Trainer, TrainingArguments)

MODEL = "neuralmind/bert-base-portuguese-cased"
LABELS = ["c1", "c234", "c5"]
LABEL_MAP = {l: i for i, l in enumerate(LABELS)}
MAX_LEN = 384

df = pd.read_excel("train.xlsx").dropna(subset=["resp_text", "clarity"])
df["resp_text"] = df["resp_text"].astype(str)
df["clarity"] = df["clarity"].astype(str).str.strip()
df = df[df["clarity"].isin(LABELS)].copy()
df["label"] = df["clarity"].map(LABEL_MAP)

# MESMO holdout de sempre
dev_df, hold_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
hold_labels = hold_df["label"].to_numpy()
print(f"dev={len(dev_df)} holdout={len(hold_df)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL)
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"GPU: {torch.cuda.get_device_name(0)} | bf16={USE_BF16}")

class DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = list(labels) if labels is not None else None
    def __getitem__(self, i):
        d = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None: d["labels"] = self.labels[i]
        return d
    def __len__(self): return len(self.enc["input_ids"])

def compute_metrics(p):
    return {"f1_macro": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")}

## O grid4 configurações. A primeira reproduz o EP1 como referência (deve dar ~0,45).Se quiser economizar tempo, rode só as 3 últimas.

In [ ]:
GRID = [
    {"tag": "ref_ep1", "lr": 2e-5, "epochs": 4, "batch": 16, "seed": 42},
    {"tag": "A",       "lr": 3e-5, "epochs": 3, "batch": 16, "seed": 42},
    {"tag": "B",       "lr": 1e-5, "epochs": 5, "batch": 16, "seed": 42},
    {"tag": "C",       "lr": 2e-5, "epochs": 2, "batch": 32, "seed": 42},
]

def run(cfg):
    tr, va = train_test_split(dev_df, test_size=0.1, stratify=dev_df["label"],
                              random_state=cfg["seed"])
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
    args = TrainingArguments(
        output_dir=f"./out_{cfg['tag']}",
        num_train_epochs=cfg["epochs"],
        per_device_train_batch_size=cfg["batch"],
        per_device_eval_batch_size=64,
        learning_rate=cfg["lr"],
        warmup_steps=500,
        weight_decay=0.01,
        fp16=not USE_BF16, bf16=USE_BF16, max_grad_norm=1.0,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="f1_macro",
        greater_is_better=True, save_total_limit=1,
        seed=cfg["seed"], report_to="none", logging_steps=200,
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=DS(tr["resp_text"], tr["label"]),
                      eval_dataset=DS(va["resp_text"], va["label"]),
                      compute_metrics=compute_metrics,
                      data_collator=DataCollatorWithPadding(tokenizer),
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()
    pr = trainer.predict(DS(hold_df["resp_text"], hold_df["label"]))
    probs = torch.softmax(torch.tensor(pr.predictions.astype("float32")), -1).numpy()
    y = probs.argmax(1)
    acc = accuracy_score(hold_labels, y); f1 = f1_score(hold_labels, y, average="macro")
    np.savez(f"probs_grid_{cfg['tag']}.npz", y=hold_labels, probs=probs)
    print(f"\n>>> {cfg['tag']}: lr={cfg['lr']} ep={cfg['epochs']} bs={cfg['batch']} "
          f"-> acc={acc:.4f} f1m={f1:.4f}")
    del model, trainer
    torch.cuda.empty_cache()
    return {**cfg, "acc": acc, "f1_macro": f1}

rows = []
for cfg in GRID:
    rows.append(run(cfg))
    pd.DataFrame(rows).to_csv("grid_bert_results.csv", index=False)   # salva a cada config
    print(pd.DataFrame(rows)[["tag","lr","epochs","batch","acc","f1_macro"]].to_string(index=False))

## Resumo e download

In [ ]:
res = pd.DataFrame(rows).sort_values("acc", ascending=False)
print(res[["tag","lr","epochs","batch","acc","f1_macro"]].to_string(index=False))
print("\nreferencias locais: BERT 1 semente=0.4500 | BERT+CNB=0.4760")

files.download("grid_bert_results.csv")
for cfg in GRID:
    f = f"probs_grid_{cfg['tag']}.npz"
    if os.path.exists(f): files.download(f)